In [115]:
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq

peptides = {
    'XBA96065.1|precursor|Mammarenavirus_lassaense|LF2350|Homo_sapiens|Sierra_Leone|2012|S|?|tile_14': 'KCNEKHDEEFCDMLRLFDFNKQAIQRLKAEAQMSIQLINKAVNALI', 
    'AZI96412.1|precursor|Mammarenavirus_lassaense|LASV/H.sapiens-wt/NGA/2018/IRR_064|Homo_sapiens|Nigeria:_EBONYI|03-Feb-2018|S|?|tile_16': 'DQLIMKNHLRDIMGIPYCNYSKYWYLNHTVTGKTSLPKCWLVSNGS', 
    'AZI96488.1|precursor|Mammarenavirus_lassaense|LASV/H.sapiens-wt/NGA/2018/IRR_135|Homo_sapiens|Nigeria:_EDDO|05-Mar-2018|S|?|tile_17': 'WYLNHTVTGKTSLPRCWLVSNGTYLNETHFSDDIEQQADNMITELL', 
    'AHC95553.1|precursor|Mammarenavirus_lassaense|?|Mastomys_natalensis|Mali|Jun-2009|S|Soromba-R|tile_19': 'TSLPKCWLVSNGSYLNETHFSDDIEQQADNMITEMLQKEYMDRQGK', 
    'AZI96059.1|precursor|Mammarenavirus_lassaense|LASV/H.sapiens-tc/NGA/2012/IRR_006|Homo_sapiens|Nigeria:_Rivers|21-Feb-2012|S|?|tile_16': 'NDQLIMKNHLRDIMGIPYCNYSKYWYLNNTVTGKSSLPKCWLVSNG', 
    'AZI96055.1|precursor|Mammarenavirus_lassaense|LASV/H.sapiens-tc/NGA/2013/IRR_002|Homo_sapiens|Nigeria:_Benue|17-Jan-2013|S|?|tile_14': 'CNEKHDEEFCDMLRLFDFNKQAIKRLKTEAQMNIQLINKAVNALIN', 
    'AHC95553.1|precursor|Mammarenavirus_lassaense|?|Mastomys_natalensis|Mali|Jun-2009|S|Soromba-R|tile_15': 'AELKCFGNTAVAKCNEKHDEEFCDMLRLFDFNKQAISRLRSEAQMS', 
    'AVV65117.1|precursor|Mammarenavirus_lassaense|Lassa_virus/H.sapiens-wt/NGA/2018/ISTH_0026|Homo_sapiens|Nigeria|07-Jan-2018|S|?|tile_18': 'YLNETYFSDDIEQQADNMITELLQKEYIDRQGRTPLGLVDLFVFST', 
    'WAD86895.1|glycoprotein|Mammarenavirus_lassaense|?|Homo_sapiens|Nigeria|1977|S|GA391|tile_14': 'CNEKHDEEFCDMLRLFDFNKQAIRRLKTEAQMSIQLINKAVNALIN', 
    #'BAL03415.1|precursor|Mammarenavirus_lunaense|?|Mastomys_natalensis|Zambia:Namwala|2009-08-18|S|NMW-1|tile_17': 'KFWYLNDTRNGKTSLPKCWKVSNNSYLNETHFSNEIEEEANNMITE'
}


### Rationale: 

### 1. Generate a DMS library from these peptides. keep wild type background for every peptide. 
### 2. Generate a 3-aa sliding alanine scan
### 3. if space allows, generate 6-aa sliding alaning scan
### 4. convert to nt (codon-optimized), add cloning linkers, remove restriction enzyme sites

### starting with 1. Generation of a DMS library from these peptides

In [116]:
amino_acids = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']

def dms_scan(peptides):
    sequences = []

    for header, peptide_seq in peptides.items():

        sequences.append({
            "header": f"{header}|WT", 
            "mutation": "WT", 
            "sequence": peptide_seq
        })

        #generate all single-AA mutants
        for i in range(0, len(peptide_seq)):
            original_aa = peptide_seq[i]

            for new_aa in amino_acids: 
                
                if new_aa == original_aa:
                    continue

                mutant_seq = peptide_seq[:i] + new_aa + peptide_seq[i+1:]

                mutation_id = f"{original_aa}{i+1}{new_aa}"

                mutant_header = f"{header}|{mutation_id}"

                sequences.append({
                    "header": mutant_header, 
                    "mutation": mutation_id, 
                    "sequence": mutant_seq
                })

        #now generate 3-aa blocks
        for i in range(0, len(peptide_seq) - 2):
            original_block = peptide_seq[i:i+3]

            #skip because identical to WT
            if original_block == 'AAA':
                continue
            
            mutant_aa_seq = peptide_seq[:i] + 'AAA' + peptide_seq[i + 3:]

            mutation_id = f"{original_block}{i+1}-{i+3}{'AAA'}"

            mutant_header = f"{header}|{mutation_id}"

            sequences.append({
                "header": mutant_header, 
                "mutation": mutation_id, 
                "sequence": mutant_aa_seq
            })


        #now generate stop-scan
        for i in range(0, len(peptide_seq)):
            original_aa = peptide_seq[i]

            mutant_aa_seq = peptide_seq[:i] +"*" + peptide_seq[i+1: ]

            mutation_id = f"{original_aa}{i+1}*"

            mutant_header = f"{header}|{mutation_id}"

            sequences.append({
                            "header": mutant_header, 
                            "mutation": mutation_id, 
                            "sequence": mutant_aa_seq
                        })

    return sequences


In [117]:
AA2NA = {
"A": list("GCT,GCC,GCA,GCG".split(",")),
"R": list("CGT,CGC,CGA,CGG,AGA,AGG".split(",")),
"N": list("AAT,AAC".split(",")),
"D": list("GAT,GAC".split(",")),
"C": list("TGT,TGC".split(",")),
"Q": list("CAA,CAG".split(",")),
"E": list("GAA,GAG".split(",")),
"G": list("GGT,GGC,GGA,GGG".split(",")),
"H": list("CAT,CAC".split(",")),
"I": list("ATT,ATC,ATA".split(",")),
"L": list("TTA,TTG,CTT,CTC,CTA,CTG".split(",")),
"K": list("AAA,AAG".split(",")),
"M": list("ATG".split(",")),
"F": list("TTT,TTC".split(",")),
"P": list("CCT,CCC,CCA,CCG".split(",")),
"S": list("TCT,TCC,TCA,TCG,AGT,AGC".split(",")),
"T": list("ACT,ACC,ACA,ACG".split(",")),
"W": list("TGG".split(",")),
"Y": list("TAT,TAC".split(",")),
"V": list("GTT,GTC,GTA,GTG".split(",")),
"*": list("TAA,TGA,TAG".split(","))}

NA2AA = {
    codon: aa
    for aa, codons in AA2NA.items()
    for codon in codons
}

In [118]:
def replace_restriction_sites(seq):
    """
    Iteratively replaces all restriction enzyme cut sites using synonymous mutations (in-frame codon)
    
    Parameters
    ----------
    seq: str
        codon-optimized nucleotide sequence 

    AA2NA : dict
        amino acid to synonymous codon mapping
    
    NA2AA : dict
        codon to amino acid mapping
    
    Returns
    -------
    new_seq: str or None
        nucleotide sequence with synonomous mutation at restriction site, or None if no restriction sites
    
    Notes
    -----
    restriction sites:
    ecoRI='GAATTC'
    hindIII='AAGCTT'
    bamHI='GGATCC'
    XhoI='CTCGAG'
    """
    restriction_sites=['GAATTC','AAGCTT','GGATCC','CTCGAG']
    max_passes = 100 
    
    for _ in range(max_passes):
        #checks if we successfully removed any site in this round, if not stop early.
        modified = False

        for site in restriction_sites: 
            #finds the first index where the site appears
            idx = seq.find(site)
            #if not found, skip and go to next one
            if idx == -1: 
                continue 

            #try mutating one overlapping codon within the site window
            for offset in range(len(site) -2): #slide window to find codon overlap
                #this snaps to the nearest codon boundary (multiple of 3)
                codon_start = (idx + offset) - ((idx + offset) % 3)
                #exract a 3-letter codon starting at that position
                codon = seq[codon_start:codon_start + 3]

                #validate codon
                if len(codon) != 3 or codon not in NA2AA: 
                    continue #malformed codon or not standard, invalid or isn't in reverse mapping so skip it. 

                #find the amino acid that the codon encodes
                aa = NA2AA[codon]
                #get all synonymous codons for that amino acid
                synonyms = AA2NA[aa]

                #skip if there are no alternatives e.g. TGG -> Tryptophan
                if len(synonyms) <= 1:
                    continue #no alternative codons for this AA

                #try each alternative codon and construct a sequence with the substituted codon
                for alt in synonyms: 
                    if alt != codon: 
                        #mutate codon and check if site is broken
                        mutated_seq = seq[:codon_start] + alt + seq[codon_start + 3:]
                        if site not in mutated_seq:
                            seq = mutated_seq
                            modified = True 
                            break 

                    if modified: 
                        break #codon successfully mutated 
                if modified: 
                    break #move on to re-scan all sites 
            if not modified:
                break

    return seq

In [119]:
import random

def aa2na(seq):
    na_seq = [
        random.choice(AA2NA.get(aa, ["---"]))
        for aa in seq
    ]
    return "".join(na_seq)

In [120]:
five_prime = "GTGGTTGGTGCTGTAGGAGCA"
three_prime = "TGATAAGCATATGCCATGGCCTC"

In [121]:
restriction_sites = [
    "GAATTC",  # EcoRI
    "AAGCTT",  # HindIII
    "GGATCC",  # BamHI
    "CTCGAG"   # XhoI
]

In [122]:
def process_dms_library(results):

    records_out = []
    unresolved_restriction = []
    translation_mismatches = []

    restriction_sites = [
        "GAATTC",
        "AAGCTT",
        "GGATCC",
        "CTCGAG"
    ]

    for entry in results:

        header = entry["header"]
        aa_seq = entry["sequence"]

        # 1. Convert AA to DNA
        nt_seq = aa2na(aa_seq)

        # 2. Add linkers
        nt_seq_with_linkers = five_prime + nt_seq + three_prime

        # 3. Remove restriction sites
        nt_seq_clean = replace_restriction_sites(nt_seq_with_linkers)

        # 4. Check that restriction sites are gone
        if any(site in nt_seq_clean for site in restriction_sites):
            unresolved_restriction.append(entry)

            continue #don't add this sequence to final library

        # 5. extract peptide coding region
        coding_seq = nt_seq_clean[
            len(five_prime):-len(three_prime)
        ]

        #6. translate as sanity check
        translated = str(
            Seq(coding_seq).translate()
        )

        if translated != aa_seq:

            translation_mismatches.append({
                "header": header, 
                "expected": aa_seq, 
                "translated": translated
            })

            #don't include failed sequence
            continue
        
        #passed everything
        record = SeqRecord(
            Seq(nt_seq_clean), 
            id=header, 
            description=""
        )

        records_out.append(record)

    
    return(
        records_out, 
        unresolved_restriction,
        translation_mismatches
    )

In [123]:
results = dms_scan(peptides)

In [124]:
with open("dms_library.fasta", 'w') as f:
    for entry in results:
        f.write(f">{entry['header']}\n")
        f.write(f"{entry['sequence']}\n")

In [125]:
(
    final_records, 
    failed_restriction, 
    failed_translation
) = process_dms_library(results)

In [126]:
#write passing sequences

output_fasta = "dms_library_final.fasta"

with open(output_fasta, "w") as out_f:
    SeqIO.write(
        final_records,
        out_f,
        "fasta"
    )

print(
    f"Done! {len(final_records)} sequences "
    f"written to {output_fasta}"
)

print(
    f"{len(failed_restriction)} sequences removed "
    f"because restriction sites could not be resolved."
)

print(
    f"{len(failed_translation)} sequences removed "
    f"because translation did not match."
)

Done! 8682 sequences written to dms_library_final.fasta
3 sequences removed because restriction sites could not be resolved.
0 sequences removed because translation did not match.


In [127]:
failed_translation

[]

In [128]:
from Bio import SeqIO

records = list(
    SeqIO.parse("dms_library_final.fasta", "fasta")
)

print("Records in FASTA:", len(records))

correct = [
    rec for rec in records
    if str(rec.seq).endswith(three_prime)
]

incorrect = [
    rec for rec in records
    if not str(rec.seq).endswith(three_prime)
]

print("Correct linker:", len(correct))
print("Incorrect linker:", len(incorrect))

Records in FASTA: 8682
Correct linker: 8682
Incorrect linker: 0
